In [1]:
!wget https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx

--2026-03-30 18:23:15--  https://github.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/raw/master/train.xlsx
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/master/train.xlsx [following]
--2026-03-30 18:23:15--  https://raw.githubusercontent.com/laxmimerit/IMDB-Movie-Reviews-Large-Dataset-50k/master/train.xlsx
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13723193 (13M) [application/octet-stream]
Saving to: ‘train.xlsx’

train.xlsx          100%[===================>]  13.09M  --.-KB/s    in 0.07s   

2026-03-30 18:23:17 (180 MB/s) - ‘train.x

In [2]:
import pandas as pd
import numpy as np
df=pd.read_excel('train.xlsx')

In [3]:
df = df.sample(500, random_state=42).reset_index(drop=True)
df

,Reviews,Sentiment
0,I really think that people are taking the wron...,pos
1,There was a stylish approach to this film on t...,pos
2,My wife and I just finished watching Bûsu AKA ...,neg
3,As with all environmentally aware films from t...,pos
4,"With an absolutely amazing cast and crew, this...",neg
...,...,...
495,"Who would think Andy Griffith's ""Helen Crump"" ...",pos
496,this has got to be one of those films where th...,neg
497,Made me think about it for days after seeing i...,pos
498,"Well, I get used after awhile to read comments...",pos


In [4]:
import torch
from collections import Counter

def tokenize(text):
  return text.lower().split()  #har bir sozni kichik harflarga keltirib har textni sozga bo'lib chiqamiz

counter = Counter(word for text in df['Reviews'] for word in tokenize(text))
# Reviews ichidagi har bir sozni tokienize qilib counter sifatida ozimizga saqlab olamiz
most_common = counter.most_common(2000) # eng kop uchraydigan sozlarni oladi
vocab = {word: i+2 for i, (word , _) in enumerate(most_common)}
vocab['<pad>'] = 0 # bo'sh joylarga 0 qoyib qaytaradi
vocab['<unk>'] = 1 # 2000 tas sozdan boshqasini 1 qaytaradi

def encode(text):
  return torch.tensor([vocab.get(word, 1) for word in tokenize(text)])

df['encode_text'] = df['Reviews'].apply(encode)
df


#gap → text
#text → lowercase
#lowercase → tokens
#tokens → numbers

,Reviews,Sentiment,encode_text
0,I really think that people are taking the wron...,pos,"[tensor(9), tensor(61), tensor(96), tensor(11)..."
1,There was a stylish approach to this film on t...,pos,"[tensor(57), tensor(14), tensor(3), tensor(170..."
2,My wife and I just finished watching Bûsu AKA ...,neg,"[tensor(53), tensor(364), tensor(4), tensor(9)..."
3,As with all environmentally aware films from t...,pos,"[tensor(15), tensor(17), tensor(34), tensor(1)..."
4,"With an absolutely amazing cast and crew, this...",neg,"[tensor(17), tensor(35), tensor(434), tensor(5..."
...,...,...,...
495,"Who would think Andy Griffith's ""Helen Crump"" ...",pos,"[tensor(38), tensor(52), tensor(96), tensor(14..."
496,this has got to be one of those films where th...,neg,"[tensor(10), tensor(42), tensor(217), tensor(6..."
497,Made me think about it for days after seeing i...,pos,"[tensor(100), tensor(94), tensor(96), tensor(4..."
498,"Well, I get used after awhile to read comments...",pos,"[tensor(313), tensor(9), tensor(92), tensor(38..."


In [5]:
from torch.utils.data import Dataset, DataLoader
class IMDBDataset(Dataset):
  def __init__(self, df):
    self.X = [encode (text) for text in df['Reviews']]
    self.y = [torch.tensor([1.0 if s=='pos' else 0.0]) for s in df['Sentiment']]

  def __len__(self):
    return len(self.X)
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

In [6]:
def collate_fn(batch):
  Xs, ys = zip(*batch)
  max_len = max(len(x) for x in Xs)
  padded_X = [torch.cat([X, torch.zeros(max_len-len(X), dtype= torch.long)]) for X in Xs]
  return torch.stack(padded_X), torch.stack(ys)

train_loader = DataLoader(IMDBDataset(df), batch_size=32, collate_fn=collate_fn)

In [7]:
import torch.nn as nn
class SentimentRNN(nn.Module):
  def __init__(self,vocab_size, embedding_dim = 64, hidden_dim = 128) -> None:
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
    self.fc = nn.Linear(hidden_dim, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x):
    x = self.embedding(x)
    _, h = self.rnn(x)
    x = self.fc(h.squeeze(0))
    return self.sigmoid(x)


In [8]:
model = SentimentRNN(len(vocab))
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [9]:
for epoch in range(10):
  total_loss = 0
  for X, y in train_loader:
    y_pred = model(X)
    loss = criterion(y_pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
  print(f'Epoch: {epoch+1}, Loss: {total_loss/len(train_loader)}')

Epoch: 1, Loss: 0.6931673921644688
Epoch: 2, Loss: 0.6807799078524113
Epoch: 3, Loss: 0.6782921440899372
Epoch: 4, Loss: 0.6758571080863476
Epoch: 5, Loss: 0.6731204949319363
Epoch: 6, Loss: 0.6724349744617939
Epoch: 7, Loss: 0.6713629327714443
Epoch: 8, Loss: 0.6736101023852825
Epoch: 9, Loss: 0.8461707979440689
Epoch: 10, Loss: 0.7190686017274857


In [10]:
def predict(text):
  model.eval()
  with torch.no_grad():
    x = encode(text)
    x =x[:100]
    if len(x) < 100:
      x = torch.cat([x, torch.zeros(100-len(x), dtype=torch.long)])
    x = x.unsqueeze(0)
    y_pred = model(x)

    prob = y_pred.item()
    label = 'positive' if prob < 0.5 else 'negative'
    print(f"ishonchi: {prob:.2f}, bashorat: {label}")

In [11]:
predict('i love this movie')

ishonchi: 0.44, bashorat: positive
